# LTV Analysis — Tidemill API

Lifetime Value, ARPU, and cohort revenue analysis from the Tidemill engine.

**API base:** `http://localhost:8000` (after `make seed && make dev`) or set `TIDEMILL_API` env var

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import requests

warnings.filterwarnings("ignore", "Unverified HTTPS")

API = os.environ.get("TIDEMILL_API", "http://localhost:8000")

START, END = "2025-09-01", "2026-03-31"

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3})


def api_get(path, **params):
    r = requests.get(f"{API}{path}", params=params)
    r.raise_for_status()
    return r.json()

## 1. Current ARPU

Average Revenue Per User = total active MRR / distinct active customers.

In [ ]:
arpu = api_get("/api/metrics/ltv/arpu")
print(f"ARPU: ${arpu / 100:,.2f}" if arpu is not None else "ARPU: N/A (no active customers)")

## 2. Monthly ARPU Timeline

Track ARPU over time to see how average revenue per customer evolves.

In [ ]:
months = pd.date_range(START, END, freq="MS")
monthly_arpu = []
for m in months:
    at = m.strftime("%Y-%m-%d")
    val = api_get("/api/metrics/ltv/arpu", at=at)
    monthly_arpu.append({"month": m.strftime("%Y-%m"), "arpu": val})

df_arpu = pd.DataFrame(monthly_arpu)
df_arpu["arpu_dollars"] = df_arpu.arpu.apply(lambda v: v / 100 if v is not None else float("nan"))

fig, ax = plt.subplots()
valid = df_arpu.dropna(subset=["arpu_dollars"])
ax.plot(range(len(valid)), valid.arpu_dollars, "o-", color="#9b59b6", linewidth=2.5, markersize=8)
ax.fill_between(range(len(valid)), valid.arpu_dollars, alpha=0.1, color="#9b59b6")

for i, (_, row) in enumerate(valid.iterrows()):
    ax.annotate(
        f"${row.arpu_dollars:,.0f}",
        (i, row.arpu_dollars),
        textcoords="offset points",
        xytext=(0, 12),
        ha="center",
        fontweight="bold",
    )

ax.set_xticks(range(len(valid)))
ax.set_xticklabels(valid.month, rotation=45)
ax.set_title("Monthly ARPU")
ax.set_ylabel("ARPU ($)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

## 3. Simple LTV

Simple LTV = ARPU / monthly logo churn rate. Uses the full period for the churn rate estimate.

In [ ]:
ltv = api_get("/api/metrics/ltv", start=START, end=END)

if ltv is not None:
    print(f"Simple LTV:  ${ltv / 100:,.2f}")
    print(f"ARPU:        ${arpu / 100:,.2f}" if arpu else "ARPU: N/A")
    if arpu and ltv:
        implied_churn = arpu / ltv
        print(f"Implied monthly churn rate: {implied_churn * 100:.1f}%")
else:
    print("LTV: N/A (insufficient data — need both active customers and churn)")

## 4. Cohort LTV

Average total revenue per customer by the month they first subscribed.

In [ ]:
cohort_ltv = api_get("/api/metrics/ltv/cohort", start=START, end=END)

if cohort_ltv:
    df_cohort = pd.DataFrame(cohort_ltv)
    df_cohort["avg_dollars"] = df_cohort.avg_revenue_per_customer.apply(
        lambda v: v / 100 if v else 0
    )
    df_cohort["total_dollars"] = df_cohort.total_revenue.apply(lambda v: v / 100 if v else 0)

    print("Cohort LTV:")
    for _, row in df_cohort.iterrows():
        print(
            f"  {row.cohort_month}:  {row.customer_count:>3} customers,"
            f"  total=${row.total_dollars:>10,.2f},"
            f"  avg=${row.avg_dollars:>8,.2f}"
        )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Average revenue per customer by cohort
    ax1.bar(range(len(df_cohort)), df_cohort.avg_dollars, color="#9b59b6", alpha=0.8)
    ax1.set_xticks(range(len(df_cohort)))
    ax1.set_xticklabels([str(c) for c in df_cohort.cohort_month], rotation=45)
    ax1.set_title("Avg Revenue per Customer by Cohort")
    ax1.set_ylabel("Revenue ($)")
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))

    for i, v in enumerate(df_cohort.avg_dollars):
        ax1.text(i, v + 2, f"${v:,.0f}", ha="center", fontweight="bold", fontsize=9)

    # Cohort sizes
    ax2.bar(range(len(df_cohort)), df_cohort.customer_count, color="#3498db", alpha=0.8)
    ax2.set_xticks(range(len(df_cohort)))
    ax2.set_xticklabels([str(c) for c in df_cohort.cohort_month], rotation=45)
    ax2.set_title("Customers per Cohort")
    ax2.set_ylabel("Count")

    for i, v in enumerate(df_cohort.customer_count):
        ax2.text(i, v + 0.2, str(v), ha="center", fontweight="bold", fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print("No cohort LTV data available yet.")
    print("This requires paid invoices to be processed via the event pipeline.")